In [ ]:
import torch
import torch.nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [ ]:
train_dataset = datasets.MNIST(root='.',
                            train=True,  
                            transform=transforms.ToTensor(),
                            download=True)
test_dataset = datasets.MNIST(root='.',
                            train=False,  
                            transform=transforms.ToTensor())

In [ ]:
batch_size = 100

train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                            batch_size=batch_size, 
                                            shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                            batch_size=batch_size, 
                                            shuffle=False)

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(RNNModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True)
    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, hn = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
input_dim = 28
hidden_dim = 100
layer_dim = 1
output_dim = 10

#가상으로 형태를 만들어봄
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  

model = RNNModel(input_dim, hidden_dim, layer_dim, output_dim).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        images = images.view(-1, input_dim, input_dim)
        optimizer.zero_grad()  
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        images = images.view(-1, 28, input_dim)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum()
    accuracy = 100 * correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%')

# Epoch [19/20], Loss: 0.1973, Accuracy: 95.63%
# Epoch [20/20], Loss: 0.0572, Accuracy: 96.03%